In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("1010_daily.csv", parse_dates=["datetime"])
print(f"Total rows : {len(df)}")
print(f"First date : {df['datetime'].min().date()}")
print(f"Last date  : {df['datetime'].max().date()}")
n = len(df)
print(f"85% cutoff : row {int(n*0.85)} → approx {df['datetime'].iloc[int(n*0.85)].date()}")

Total rows : 5544
First date : 2004-02-24
Last date  : 2026-05-13
85% cutoff : row 4712 → approx 2023-01-10


In [3]:
import pickle, numpy as np

with open("models/1010/1010_target_scaler.pkl", "rb") as f:
    ts = pickle.load(f)

print("target mean  :", ts.mean_)    # [price_mean, vol_mean]
print("target scale :", ts.scale_)   # [price_std,  vol_std]

target mean  : [ 0.02451117 -0.03112414]
target scale : [1.33884711 0.69474484]


In [ ]:
#special case company
"""
"4348"
'2285', '4081', '2286', '4340', '4342', '2283', 
'2284', '1323', '4009', '4334', '4084', '4261', 
'4012', '1830', '4333', '1835', '4165', '4347', 
'2382', '2223', '4163', '4262', '4083', '4164', 
'1834', '1303', '2081', '4162', '7204', '7202', 
'2281', '1833', '4143', '4142', '4263', '1831', 
'4051', '1322', '1182', '6017', '4161', '4031', 
'4071', '4322', '4264', '4016', '4072', '4191', 
'4332', '2381', '6014', '4017', '4015', '4324', 
'6015', '4336', '4338', '4082', '4144', '4014', 
'1111', '4291', '4339', '4013', '2282', '4325', 
'1304', '7203', '4018', '2084', '7200', '2287', 
'4331', '8313', '4193', '1183', '4192', '4345', 
'4344', '6016', '2082', '2083', '6013', '3007',
# collapse problem
"8200", "1050","1030", "1150", "4003", "2020", "1140","7020"


"""

# done training
"""
"8300","1010", "8100","2050", "2060", "1180", "3080", "3090"

2nd batch
 "3030", "1020",
 
"""



symbols = [


    "2150", "4260",
    # "3020", "4001", "1320", "4007", "3040",
    # "8280", "6070", "3010", "2270", "3003", "1120", "8170",
    # "1214", "8010", "3002", "3050", "8030", "4250", "3005",
    # "2230", "4280", "4020", "4004", "8210", "2280", "1212",
    # "4002", "8012", "4150", "8160", "8070", "3004", "2080",
    # "3060", "1302", "8020", "4300", "4090", "6001", "6004",
    # "7040", "2310", "4006", "2300", "8230", "4200", "4100",
    # "2070", "2190", "2250", "1211", "8180", "2290", "2240",
    # "6020", "1301", "4180", "8240", "4210", "8040", "4130",
    # "8310", "4290", "4050", "4040", "1210", "4080", "2010",
    # "4170", "4240"
    ]



dfs = {}

for s in symbols:
    filepath = f"{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── 2150 ──────────────────────────
  Shape : (6089, 46)
  Head  :
              datetime  ticker  open_price  high_price  low_price  \
0  1994-11-20 16:00:00    2150   21.066667   21.066667  20.133333   
1  1994-11-21 16:00:00    2150   20.933333   20.933333  20.933333   
2  1994-12-20 16:00:00    2150   21.333333   21.333333  20.666667   
3  1994-12-21 16:00:00    2150   20.000000   20.000000  19.333333   
4  1995-01-04 16:00:00    2150   19.333333   19.333333  19.333333   

   close_price   volume     EMA_10     EMA_20  EMA_ratio  ...  volume_lag_3  \
0    20.133333  15000.0  21.631966  22.344966   0.901023  ...       14250.0   
1    20.933333  15000.0  21.504942  22.210525   0.942496  ...       30000.0   
2    20.666667  14632.5  21.352528  22.063491   0.936691  ...        7500.0   
3    19.333333   1500.0  20.985402  21.803476   0.886709  ...       15000.0   
4    19.333333   5992.5  20.685026  21.568224   0.896380  ...       15000.0   

   volume_lag_4  volume_lag_5  above_ema10 

In [3]:
for symbol in symbols:
    df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    
    # add your lag features + volume_pct_change_log here same as your pipeline
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # Regime and momentum persistence features
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)
    daily_up = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.dropna().reset_index(drop=True)

    # outlier removal
    for col, sigma in {'price_pct_change': 3, 'volume_pct_change_log': 2}.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # split
    n = len(df)
    train_df = df.iloc[:int(n * 0.70)]
    val_df   = df.iloc[int(n * 0.70):int(n * 0.85)]

    # scale
    target_scaler = StandardScaler()
    train_y = target_scaler.fit_transform(train_df[['price_pct_change', 'volume_pct_change_log']])
    val_y   = target_scaler.transform(val_df[['price_pct_change', 'volume_pct_change_log']])

    print(f"\n{symbol}")
    print(f"  Raw train UP%      : {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Raw val UP%        : {(val_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scaled train mean  : {train_y[:, 0].mean():.4f}")  # should be ~0
    print(f"  Scaled train std   : {train_y[:, 0].std():.4f}")   # should be ~1
    print(f"  Scaled val mean    : {val_y[:, 0].mean():.4f}")    # if far from 0, val is different regime
    print(f"  Scaled val std     : {val_y[:, 0].std():.4f}")
    print(f"  Train date range   : {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Val date range     : {val_df['datetime'].min().date()} → {val_df['datetime'].max().date()}")


2150
  Raw train UP%      : 47.03%
  Raw val UP%        : 46.11%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : 0.0263
  Scaled val std     : 0.8410
  Train date range   : 1995-01-17 → 2019-03-21
  Val date range     : 2019-03-24 → 2022-10-19

4260
  Raw train UP%      : 45.65%
  Raw val UP%        : 46.75%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : 0.0339
  Scaled val std     : 0.8991
  Train date range   : 2007-11-03 → 2020-12-16
  Val date range     : 2020-12-17 → 2023-08-21


In [ ]:
"""
Inference script — BiLSTM Attention ONNX models
Runs on the last 15% (test split) of each symbol's daily CSV.

File layout expected (relative to inference.py):
  {symbol}_daily.csv
  models/{symbol}/{symbol}_feature_scaler.pkl
  models/{symbol}/{symbol}_target_scaler.pkl
  models/{symbol}/{symbol}_bilstm.onnx

Output:
  models/{symbol}/{symbol}_inference_result.csv
"""

import os
import pickle
import numpy as np
import pandas as pd
import onnxruntime as ort

# ── Config ────────────────────────────────────────────────────────────────────
SYMBOLS = ["8300", "1010", "8100", "2050", "2060", "1180", "3080", "3090", "2370"]

DATA_DIR = os.path.dirname(os.path.abspath(__file__))

SEQ_LEN_DEFAULT = 30
SEQ_LEN = {
    "8300": SEQ_LEN_DEFAULT,
    "1010": SEQ_LEN_DEFAULT,
    "8100": SEQ_LEN_DEFAULT,
    "2050": SEQ_LEN_DEFAULT,
    "2060": SEQ_LEN_DEFAULT,
    "1180": SEQ_LEN_DEFAULT,
    "3080": SEQ_LEN_DEFAULT,
    "3090": SEQ_LEN_DEFAULT,
    "2370": SEQ_LEN_DEFAULT,
}

FEATURE_COLS = [
    # OHLCV (5)
    'open_price', 'high_price', 'low_price', 'close_price', 'volume',
    # Trend (4)
    'EMA_10', 'EMA_20', 'EMA_ratio', 'MACD_hist',
    # Momentum (4)
    'RSI_14', 'ROC_5', 'ROC_10', 'ROC_20',
    # Volatility (4)
    'ATR_14', 'ATR_ratio', 'BB_pct', 'realized_vol',
    # Volume (5)
    'OBV', 'OBV_momentum', 'volume_ratio', 'volume_surge', 'MFI_14',
    # Lag features (15)
    'close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_4', 'close_lag_5',
    'returns_lag_1', 'returns_lag_2', 'returns_lag_3', 'returns_lag_4', 'returns_lag_5',
    'volume_lag_1', 'volume_lag_2', 'volume_lag_3', 'volume_lag_4', 'volume_lag_5',
    # Regime / persistence (5)
    'above_ema10', 'above_ema20', 'roc5_pos', 'roc20_pos', 'up_streak',
]  # 42 total


# ── Feature engineering ───────────────────────────────────────────────────────
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    close  = df["close_price"]
    high   = df["high_price"]
    low    = df["low_price"]
    volume = df["volume"]

    # ── Trend ─────────────────────────────────────────────────────────────────
    df["EMA_10"]    = close.ewm(span=10, adjust=False).mean()
    df["EMA_20"]    = close.ewm(span=20, adjust=False).mean()
    df["EMA_ratio"] = df["EMA_10"] / df["EMA_20"]

    ema12           = close.ewm(span=12, adjust=False).mean()
    ema26           = close.ewm(span=26, adjust=False).mean()
    macd_line       = ema12 - ema26
    macd_signal     = macd_line.ewm(span=9, adjust=False).mean()
    df["MACD_hist"] = macd_line - macd_signal

    # ── Momentum ──────────────────────────────────────────────────────────────
    delta     = close.diff()
    gain      = delta.clip(lower=0).rolling(14).mean()
    loss      = (-delta.clip(upper=0)).rolling(14).mean()
    rs        = gain / loss.replace(0, np.nan)
    df["RSI_14"] = 100 - (100 / (1 + rs))

    df["ROC_5"]  = close.pct_change(5)  * 100
    df["ROC_10"] = close.pct_change(10) * 100
    df["ROC_20"] = close.pct_change(20) * 100

    # ── Volatility ────────────────────────────────────────────────────────────
    tr              = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low  - close.shift()).abs(),
    ], axis=1).max(axis=1)
    df["ATR_14"]    = tr.rolling(14).mean()
    df["ATR_ratio"] = df["ATR_14"] / close

    rolling_std     = close.rolling(20).std()
    sma20           = close.rolling(20).mean()
    bb_upper        = sma20 + 2 * rolling_std
    bb_lower        = sma20 - 2 * rolling_std
    df["BB_pct"]    = (close - bb_lower) / (bb_upper - bb_lower).replace(0, np.nan)

    df["realized_vol"] = close.pct_change().rolling(20).std() * np.sqrt(252)

    # ── Volume ────────────────────────────────────────────────────────────────
    direction       = np.sign(close.diff()).fillna(0)
    df["OBV"]       = (direction * volume).cumsum()
    df["OBV_momentum"] = df["OBV"].diff(5)
    df["volume_ratio"] = volume / volume.rolling(20).mean()
    df["volume_surge"] = (volume > volume.rolling(20).mean() * 1.5).astype(float)

    # MFI
    typical_price   = (high + low + close) / 3
    raw_money_flow  = typical_price * volume
    pos_flow        = raw_money_flow.where(typical_price > typical_price.shift(), 0).rolling(14).sum()
    neg_flow        = raw_money_flow.where(typical_price < typical_price.shift(), 0).rolling(14).sum()
    mfi_ratio       = pos_flow / neg_flow.replace(0, np.nan)
    df["MFI_14"]    = 100 - (100 / (1 + mfi_ratio))

    # ── Lag features ──────────────────────────────────────────────────────────
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = close.shift(lag)
        df[f"returns_lag_{lag}"] = close.pct_change(1).shift(lag) * 100
        df[f"volume_lag_{lag}"]  = volume.shift(lag)

    # ── Regime / persistence ──────────────────────────────────────────────────
    df["above_ema10"] = (close > df["EMA_10"]).astype(float)
    df["above_ema20"] = (close > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"]  > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)

    daily_up          = close.diff(1).gt(0).astype(int)
    streak_id         = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"]   = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up

    # ── Targets ───────────────────────────────────────────────────────────────
    df["price_pct_change"]      = close.pct_change(1) * 100
    df["volume_log"]            = np.log1p(volume)
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)

    df = df.dropna().reset_index(drop=True)
    return df


# ── Sequence builder ──────────────────────────────────────────────────────────
def create_sequences(X_scaled: np.ndarray, seq_len: int) -> np.ndarray:
    seqs = []
    for i in range(len(X_scaled) - seq_len):
        seqs.append(X_scaled[i : i + seq_len])
    return np.array(seqs, dtype=np.float32)


# ── Main inference loop ───────────────────────────────────────────────────────
any_success = False

for symbol in SYMBOLS:
    model_dir = os.path.join(DATA_DIR, "models", symbol)
    csv_path  = os.path.join(DATA_DIR, f"{symbol}_daily.csv")
    feat_pkl  = os.path.join(model_dir, f"{symbol}_feature_scaler.pkl")
    tgt_pkl   = os.path.join(model_dir, f"{symbol}_target_scaler.pkl")
    onnx_path = os.path.join(model_dir, f"{symbol}_bilstm.onnx")

    missing = [p for p in [csv_path, feat_pkl, tgt_pkl, onnx_path] if not os.path.exists(p)]
    if missing:
        print(f"[{symbol}] Skipping — missing files:")
        for p in missing:
            print(f"    {p}")
        continue

    print(f"\n{'─'*55}")
    print(f"  Inference — {symbol}")
    print(f"{'─'*55}")

    # ── Load & engineer features ──────────────────────────────────────────────
    df = pd.read_csv(csv_path, parse_dates=["datetime"])
    df = build_features(df)

    # ── Test split: last 15% ──────────────────────────────────────────────────
    n       = len(df)
    test_df = df.iloc[int(n * 0.85):].reset_index(drop=True)
    print(f"  Test rows  : {len(test_df)}")
    print(f"  Date range : {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")

    # ── Load scalers ──────────────────────────────────────────────────────────
    with open(feat_pkl, "rb") as f:
        feature_scaler = pickle.load(f)
    with open(tgt_pkl, "rb") as f:
        target_scaler  = pickle.load(f)

    # ── Scale — pass DataFrame so column names match ──────────────────────────
    feat_cols_present = [c for c in FEATURE_COLS if c in test_df.columns]
    missing_cols      = [c for c in FEATURE_COLS if c not in test_df.columns]
    if missing_cols:
        print(f"  ⚠ Missing cols (will be skipped): {missing_cols}")

    X_test = feature_scaler.transform(test_df[feat_cols_present])  # DataFrame, keeps names

    # ── Build sequences ───────────────────────────────────────────────────────
    seq_len   = SEQ_LEN[symbol]
    X_seq     = create_sequences(X_test, seq_len)
    pred_rows = test_df.iloc[seq_len:].reset_index(drop=True)
    print(f"  Sequences  : {len(X_seq)}")

    # ── ONNX inference ────────────────────────────────────────────────────────
    sess        = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    input_name  = sess.get_inputs()[0].name
    output_name = sess.get_outputs()[0].name

    raw_preds  = sess.run([output_name], {input_name: X_seq})[0]
    preds_orig = target_scaler.inverse_transform(raw_preds)

    # ── Build result table ────────────────────────────────────────────────────
    close_vals = pred_rows["close_price"].values
    pred_pct   = preds_orig[:, 0]

    result = pd.DataFrame({
        "Symbol"              : symbol,
        "Date"                : pred_rows["datetime"].dt.date.values,
        "Open"                : pred_rows["open_price"].values,
        "High"                : pred_rows["high_price"].values,
        "Low"                 : pred_rows["low_price"].values,
        "Close (Actual Price)": close_vals,
        "Actual Price pct"    : pred_rows["price_pct_change"].values,
        "Predicted Price pct" : pred_pct,
        "Predicted Price"     : close_vals * (1 + pred_pct / 100),
    })

    # ── Sanity stats ──────────────────────────────────────────────────────────
    actual    = result["Actual Price pct"].values
    predicted = result["Predicted Price pct"].values
    dir_acc   = np.mean(np.sign(actual) == np.sign(predicted))
    print(f"  Direction accuracy : {dir_acc:.2%}")
    print(f"  Pred range         : [{predicted.min():.4f}, {predicted.max():.4f}]")
    print(f"  Actual range       : [{actual.min():.4f},  {actual.max():.4f}]")

    # ── Save ──────────────────────────────────────────────────────────────────
    out_path = os.path.join(model_dir, f"{symbol}_inference_result.csv")
    result.to_csv(out_path, index=False)
    print(f"  ✓ Saved → {out_path}")
    any_success = True

if not any_success:
    print("\nNo results — check your file paths and symbol list.")

Total features: 42

──────────────────────────────────────────────────
Processing 2150...
  Rows after lag + dropna: 6,082
  Rows after outlier removal: 5,915
  Train : 4,140 (1995-01-17 → 2019-03-21)
  Val   : 887 (2019-03-24 → 2022-10-19)
  Test  : 888 (2022-10-20 → 2026-05-12)
  X_train: (4110, 30, 42) | X_val: (857, 30, 42) | X_test: (858, 30, 42)

──────────────────────────────────────────────────
Processing 4260...
  Rows after lag + dropna: 4,619
  Rows after outlier removal: 4,304
  Train : 3,012 (2007-11-03 → 2020-12-16)
  Val   : 646 (2020-12-17 → 2023-08-21)
  Test  : 646 (2023-08-22 → 2026-05-12)
  X_train: (2982, 30, 42) | X_val: (616, 30, 42) | X_test: (616, 30, 42)

── Done: 2/4 tickers prepared ──


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32) \
                 if weights is not None \
                 else torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


BATCH_SIZE = 32
loaders    = {}  # loaders["4030"] → {train, val, test}

for symbol, data in prepared.items():
    train_loader = DataLoader(TadawulDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TadawulDataset(data["X_val"],   data["y_val"]),   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(TadawulDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "val":   val_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Val: {len(val_loader)} batches | Test: {len(test_loader)} batches")

    # ── Sanity check ──────────────────────────────────────────────────────────
    sample_X, sample_y, sample_w = next(iter(train_loader))   # ← unpack 3
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 2)")
    print(f"  Sample w: {sample_w.shape} → (batch,)  weights all 1.0 since no weights passed")


2150 — Train: 129 batches | Val: 27 batches | Test: 27 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 2]) → (batch, 2)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed

4260 — Train: 94 batches | Val: 20 batches | Test: 20 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 2]) → (batch, 2)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed


In [ ]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score

# done training
"""
"8300","1010", "8100","2050", "2060", "1180", "3080", "3090",
"""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}


# ── BiLSTM + Attention Model (with LayerNorm) ─────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        # Attention with positional bias — later timesteps are more recent
        self.attention      = nn.Linear(hidden_size * 2, 1)
        self.pos_bias       = nn.Parameter(torch.zeros(1))  # learnable recency bias
        self.ln             = nn.LayerNorm(hidden_size * 2)
        self.dropout        = nn.Dropout(dropout)
        self.fc             = nn.Linear(hidden_size * 2, 2)
        
        # NEW: direct skip connection from last hidden to output
        self.skip_fc        = nn.Linear(hidden_size * 2, 2)
        self.output_blend   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)                  # (B, T, H*2)
        seq_len      = lstm_out.size(1)
        
        # Recency-biased attention
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        
        out_attn     = self.fc(self.dropout(self.ln(context)))
        
        # Skip: use last timestep directly (most recent information)
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        
        # Blend — learnable, starts 50/50
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


# ── Joint Loss: Huber (magnitude) + Directional + Std-collapse penalty ────────
class JointPredictionLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=0.5, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma  # NEW: magnitude-confidence penalty

    def forward(self, preds, targets, weights=None):
        # Huber for magnitude
        huber = nn.HuberLoss(delta=1.0, reduction="none")(preds, targets).mean(dim=1)

        # Directional loss — wrong sign with MAGNITUDE scaling
        # A prediction of -0.001 on +2.0 is penalized MORE than -0.001 on +0.1
        price_sign_loss  = torch.clamp(-preds[:, 0] * targets[:, 0], min=0) * targets[:, 0].abs()
        volume_sign_loss = torch.clamp(-preds[:, 1] * targets[:, 1], min=0) * targets[:, 1].abs()

        # Variance collapse penalty — directly penalize low prediction variance
        pred_std  = preds[:, 0].std() + 1e-8
        tgt_std   = targets[:, 0].std() + 1e-8
        std_ratio = pred_std / tgt_std
        # Penalize heavily when std_ratio < 0.3, ease off once > 0.7
        collapse_loss = torch.clamp(0.5 - std_ratio, min=0) ** 2

        # Bias penalty
        bias_penalty = preds[:, 0].mean() ** 2   # squared, not abs — smoother gradient

        per_sample = (
            huber
            + self.alpha * price_sign_loss
            + 0.3        * volume_sign_loss
        )
        if weights is not None:
            per_sample = per_sample * weights

        return (
            per_sample.mean()
            + self.beta  * std_ratio.reciprocal().clamp(max=10.0)  # NEW: reciprocal penalty
            + self.gamma * collapse_loss * 5.0
            + 2.0        * bias_penalty
        )
        
# ── Helpers ───────────────────────────────────────────────────────────────────
def create_sequences(X, y, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        move   = abs(y[i + seq_len, 0])
        ws.append(1.0 + 2.0 * float(move > move_threshold))
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


def make_balanced_sampler(y, oversample_factor=2.0):
    labels = (y[:, 0] > 0).astype(int)
    moves  = np.abs(y[:, 0])
    
    # Stratify into 4 buckets: UP-small, UP-large, DOWN-small, DOWN-large
    median_move = np.median(moves)
    bucket = labels * 2 + (moves > median_move).astype(int)
    # bucket: 0=DOWN-small, 1=DOWN-large, 2=UP-small, 3=UP-large
    
    counts = np.bincount(bucket, minlength=4).astype(float)
    counts = np.maximum(counts, 1)
    
    # Inverse frequency weights, with extra boost on large moves
    weight_map = 1.0 / counts
    weight_map[1] *= oversample_factor   # DOWN-large
    weight_map[3] *= oversample_factor   # UP-large
    
    sample_weights = torch.tensor(weight_map[bucket], dtype=torch.float32)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


def safe_corr(a, b):
    """Pearson r, returns 0.0 on NaN / zero-std edge cases."""
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else float(r)


# ── Clean NaN values before Optuna ───────────────────────────────────────────
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    for split in ("train", "val"):
        X_key, y_key = f"{split}_X_scaled", f"{split}_y_scaled"
        X, y         = data[X_key], data[y_key]
        mask         = ~(np.isnan(X).any(axis=1) | np.isnan(y).any(axis=1))
        before       = len(X)
        data[X_key]  = X[mask]
        data[y_key]  = y[mask]
        print(f"  {symbol} {split}: {before} → {len(data[X_key])} rows")


# ── Optuna search ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Optuna search — {symbol}")
    print(f"{'═'*55}")

    train_X    = data["train_X_scaled"]
    train_y    = data["train_y_scaled"]
    val_X      = data["val_X_scaled"]
    val_y      = data["val_y_scaled"]
    input_size = train_X.shape[1]

    def objective(trial):
        # ── Hyperparameters ───────────────────────────────────────────────────
        hidden_size = trial.suggest_categorical("hidden_size", [64, 128])
        num_layers  = trial.suggest_int("num_layers", 2, 3)
        dropout     = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
        lr          = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        batch_size  = trial.suggest_categorical("batch_size", [32, 64])
        seq_len     = trial.suggest_categorical("seq_len", [10, 20, 30, 40])
        alpha       = trial.suggest_float("alpha", 3.0, 12.0, step=0.5)
        beta        = trial.suggest_float("beta",  0.3, 1.0, step=0.1)
        move_threshold = trial.suggest_float("move_threshold", 0.2, 0.5, step=0.1)

        # ── Sequences (now returns weights) ───────────────────────────────────
        X_tr, y_tr, w_tr = create_sequences(train_X, train_y, seq_len, move_threshold)
        X_vl, y_vl, _    = create_sequences(val_X,   val_y,   seq_len, move_threshold)

        sampler  = make_balanced_sampler(y_tr)
        train_ld = DataLoader(
            TadawulDataset(X_tr, y_tr, w_tr),
            batch_size=batch_size,
            sampler=sampler,
            drop_last=True
        )
        val_ld = DataLoader(
            TadawulDataset(X_vl, y_vl),          # no weights needed for val
            batch_size=batch_size,
            shuffle=False,
            drop_last=False
        )

        # ── Model & optimiser ─────────────────────────────────────────────────
        model = StockPctChangeBiLSTMAttention(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout
        ).to(device)

        criterion = JointPredictionLoss(alpha=alpha, beta=beta)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        

        EPOCHS         = 30
        PATIENCE       = 7
        best_val_score = 0.0
        patience_ctr   = 0

        for epoch in range(1, EPOCHS + 1):
            # ── Train ─────────────────────────────────────────────────────────
            model.train()
            for X_batch, y_batch, w_batch in train_ld:   # ← unpack 3 items now
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                w_batch = w_batch.to(device)
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch, w_batch)  # ← pass weights
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            # ── Validate ──────────────────────────────────────────────────────
            model.eval()
            all_preds, all_actuals = [], []
            with torch.no_grad():
                for X_batch, y_batch, _ in val_ld:        # ← unpack 3 items
                    all_preds.append(model(X_batch.to(device)).cpu().numpy())
                    all_actuals.append(y_batch.numpy())

            if len(all_preds) == 0:
                return 0.0

            val_preds   = np.vstack(all_preds)
            val_actuals = np.vstack(all_actuals)

            # ── Direction (balanced accuracy) ─────────────────────────────────
            price_dir = balanced_accuracy_score(
                np.sign(val_actuals[:, 0]).astype(int),
                np.sign(val_preds[:, 0]).astype(int)
            )
            volume_dir = balanced_accuracy_score(
                np.sign(val_actuals[:, 1]).astype(int),
                np.sign(val_preds[:, 1]).astype(int)
            )

            # ── Magnitude correlation ─────────────────────────────────────────
            price_corr  = safe_corr(val_actuals[:, 0], val_preds[:, 0])
            volume_corr = safe_corr(val_actuals[:, 1], val_preds[:, 1])

            # ── Std-ratio penalty ─────────────────────────────────────────────
            pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]), 1e-6)
            std_penalty    = abs(1.0 - pred_std_ratio) * 0.2

            # ── Collapse penalty ──────────────────────────────────────────────
            pred_up_pct      = (val_preds[:, 0] > 0).mean()
            
            actual_up_pct    = (val_actuals[:, 0] > 0).mean()
            bias_gap         = abs(pred_up_pct - actual_up_pct)
            # Disqualify any trial where the model nearly always predicts one direction
            if pred_up_pct < 0.15 or pred_up_pct > 0.85:
                return -1.0   # hard disqualification, not just a penalty

            # Std collapse check — require pred std to be at least 20% of actual std
            pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]), 1e-6)
            if pred_std_ratio < 0.20:
                return -1.0   # also disqualify

            val_score = (
                0.40 * price_dir             +
                0.20 * volume_dir            +
                0.25 * max(price_corr, 0.0)  +
                0.15 * max(volume_corr, 0.0) -
                abs(1.0 - pred_std_ratio) * 0.3  -
                bias_gap * 3.0               # increased weight
            )
            trial.report(val_score, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if val_score > best_val_score:
                best_val_score = val_score
                patience_ctr   = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    break

        return best_val_score

    # ── Run ───────────────────────────────────────────────────────────────────
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        study_name=f"bilstm_attention_{symbol}"
    )
    study.optimize(objective, n_trials=50, timeout=3600)

    best_params_all[symbol] = study.best_params

    # ── Results ───────────────────────────────────────────────────────────────
    print(f"\n  Best joint score : {study.best_value:.4f}")
    print(f"  Best trial       : {study.best_trial.number}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k:15s} : {v}")

    print(f"\n  Top 5 Trials:")
    trials_df = study.trials_dataframe().sort_values("value", ascending=False).head()
    cols = [
        "number", "value",
        "params_hidden_size", "params_num_layers", "params_dropout",
        "params_lr", "params_batch_size", "params_seq_len",
        "params_alpha", "params_beta", "params_move_threshold",
    ]
    print(trials_df[[c for c in cols if c in trials_df.columns]])

    print(f"\n── Optuna done for {len(best_params_all)} symbols ──")
    print(trials_df[[c for c in cols if c in trials_df.columns]])

    print(f"\n── Optuna done for {len(best_params_all)} symbols ──")

Using device: cuda

Cleaning NaN values from all symbols...


[I 2026-05-16 13:32:15,086] A new study created in memory with name: bilstm_attention_2150


  2150 train: 4140 → 4140 rows
  2150 val: 887 → 887 rows
  4260 train: 3012 → 3012 rows
  4260 val: 646 → 646 rows

═══════════════════════════════════════════════════════
  Optuna search — 2150
═══════════════════════════════════════════════════════


[I 2026-05-16 13:32:27,117] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 1.3, 'gamma': 0.75, 'move_threshold': 0.2}. Best is trial 0 with value: -1.0.
[I 2026-05-16 13:32:41,098] Trial 1 finished with value: 0.07431426524711446 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.9000000000000001, 'gamma': 1.5, 'move_threshold': 0.2}. Best is trial 1 with value: 0.07431426524711446.
[I 2026-05-16 13:32:49,061] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'gamma': 2.0, 'move_threshold': 0.30000000000000004}. Best is trial 1 with value: 0.07431426524711446.
[I 2026-05-


  Best score : 0.2048
  Best trial : 18
  Best params:
    hidden_size     : 64
    num_layers      : 2
    dropout         : 0.2
    lr              : 0.0017427913295284558
    batch_size      : 64
    seq_len         : 20
    alpha           : 6.0
    beta            : 1.5
    gamma           : 1.75
    move_threshold  : 0.4

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
18      18  0.204752                  64                  2             0.2   
17      17  0.201331                 128                  2             0.2   
19      19  0.199321                  64                  2             0.2   
49      49  0.194378                  64                  2             0.2   
34      34  0.191702                  64                  2             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
18   0.001743                 64              20           6.0          1.5   
17   0.001363       

[I 2026-05-16 13:42:23,455] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 1.3, 'gamma': 0.75, 'move_threshold': 0.2}. Best is trial 0 with value: -1.0.
[I 2026-05-16 13:42:30,440] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.9000000000000001, 'gamma': 1.5, 'move_threshold': 0.2}. Best is trial 0 with value: -1.0.
[I 2026-05-16 13:42:36,606] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'gamma': 2.0, 'move_threshold': 0.30000000000000004}. Best is trial 0 with value: -1.0.
[I 2026-05-16 13:42:43,319] Trial 3 finished with value:


  Best score : -0.0043
  Best trial : 19
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.30000000000000004
    lr              : 0.00010272842311997562
    batch_size      : 32
    seq_len         : 40
    alpha           : 4.0
    beta            : 1.4000000000000001
    gamma           : 0.75
    move_threshold  : 0.30000000000000004

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
19      19 -0.004318                 128                  3             0.3   
21      21 -0.450671                 128                  3             0.3   
40      40 -0.470305                 128                  3             0.2   
0        0 -1.000000                 128                  3             0.3   
36      36 -1.000000                  64                  2             0.3   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
19   0.000103                 32              40 

In [7]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import os
import time

trained_models = {}


# ── Validation metrics: direction + magnitude ─────────────────────────────────
def compute_val_metrics(model, loader, device):
    """
    Returns
    -------
    joint_score  : float  — same formula as Optuna objective
    price_dir    : float  — balanced accuracy, price direction
    volume_dir   : float  — balanced accuracy, volume direction
    price_corr   : float  — Pearson r, price magnitude
    volume_corr  : float  — Pearson r, volume magnitude
    pred_up_pct  : float  — fraction of UP predictions (collapse monitor)
    std_ratio    : float  — pred_std / actual_std  (1.0 = perfect)
    """
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in loader:    # ← unpack 3
            all_preds.append(model(X_batch.to(device)).cpu().numpy())
            all_actuals.append(y_batch.numpy())

    preds   = np.vstack(all_preds)
    actuals = np.vstack(all_actuals)

    # ── Directional (balanced accuracy) ──────────────────────────────────────
    price_dir  = balanced_accuracy_score(
        np.sign(actuals[:, 0]).astype(int),
        np.sign(preds[:, 0]).astype(int)
    )
    volume_dir = balanced_accuracy_score(
        np.sign(actuals[:, 1]).astype(int),
        np.sign(preds[:, 1]).astype(int)
    )

    # ── Magnitude correlation (Pearson r) ─────────────────────────────────────
    def safe_corr(a, b):
        if np.std(a) < 1e-8 or np.std(b) < 1e-8:
            return 0.0
        r = np.corrcoef(a, b)[0, 1]
        return 0.0 if np.isnan(r) else float(r)

    price_corr  = safe_corr(actuals[:, 0], preds[:, 0])
    volume_corr = safe_corr(actuals[:, 1], preds[:, 1])

    # ── Std-ratio and collapse ────────────────────────────────────────────────
    pred_up_pct  = (preds[:, 0] > 0).mean()
    std_ratio    = np.std(preds[:, 0]) / max(np.std(actuals[:, 0]), 1e-6)
    std_penalty  = abs(1.0 - std_ratio) * 0.2
    collapse_penalty = 1.0 if (pred_up_pct < 0.05 or pred_up_pct > 0.95) else 0.0
    
    

    # ── Joint score (mirrors Optuna objective exactly) ────────────────────────
    joint_score = (
        0.35 * price_dir               +
        0.25 * volume_dir              +
        0.25 * max(price_corr,  0.0)   +
        0.15 * max(volume_corr, 0.0)   -
        std_penalty                    -
        collapse_penalty
    )
    

    return joint_score, price_dir, volume_dir, price_corr, volume_corr, pred_up_pct, std_ratio


# ── Training loop ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*72}")
    print(f"  Training — {symbol}")
    print(f"{'═'*72}")

    best = best_params_all[symbol]
    print(f"  Best params: {best}")

    move_threshold = best.get("move_threshold", 0.3)   # fallback if not in best_params

    # ── Rebuild sequences with best seq_len ───────────────────────────────────
    X_train_f, y_train_f, w_train_f = create_sequences(data["train_X_scaled"], data["train_y_scaled"], best["seq_len"], move_threshold)
    X_val_f,   y_val_f,   _         = create_sequences(data["val_X_scaled"],   data["val_y_scaled"],   best["seq_len"], move_threshold)
    X_test_f,  y_test_f,  _         = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)

    # ── Loaders ───────────────────────────────────────────────────────────────
    sampler      = make_balanced_sampler(y_train_f)
    train_loader = DataLoader(
        TadawulDataset(X_train_f, y_train_f, w_train_f),   # ← pass weights
        batch_size=best["batch_size"],
        sampler=sampler,
        drop_last=True
    )
    val_loader  = DataLoader(TadawulDataset(X_val_f,  y_val_f),  batch_size=best["batch_size"], shuffle=False)
    test_loader = DataLoader(TadawulDataset(X_test_f, y_test_f), batch_size=best["batch_size"], shuffle=False)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"]
    ).to(device)

    WARMUP_EPOCHS    = 10
    BASE_ALPHA       = best["alpha"]
    BASE_BETA        = best["beta"]
    BASE_GAMMA       = best.get("gamma", 1.0)   # fallback for old best_params
    optimizer        = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
    scheduler        = ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)
    #                                          ↑ 'max' — we track joint_score, not loss

    EPOCHS           = 100
    PATIENCE         = 15
    best_joint_score = -np.inf
    patience_ctr     = 0
    history          = {"train_loss": [], "val_loss": [], "joint_score": [], "price_dir": [], "price_corr": []}

    os.makedirs(f"models/{symbol}", exist_ok=True)
    save_path = f"models/{symbol}/{symbol}_best_bilstm.pt"

    print(f"\n{'Epoch':>6} | {'TrLoss':>8} | {'VaLoss':>8} | "
          f"{'Joint':>7} | {'PDir':>6} | {'VDir':>6} | "
          f"{'Pr':>6} | {'Vr':>6} | {'UP%':>5} | {'StdR':>5} | {'s':>5}")
    print("─" * 90)
    
    training_state = type('State', (), {})() 
    
    for epoch in range(1, EPOCHS + 1):
        start        = time.time()
        warmup_scale = min(1.0, epoch / WARMUP_EPOCHS)
        criterion    = JointPredictionLoss(
            alpha = BASE_ALPHA * warmup_scale,
            beta  = BASE_BETA,
            gamma = BASE_GAMMA
        )

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for X_batch, y_batch, w_batch in train_loader:
            X_batch, y_batch, w_batch = X_batch.to(device), y_batch.to(device), w_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # ── Val loss ──────────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch, _ in val_loader:
                val_losses.append(
                    criterion(model(X_batch.to(device)), y_batch.to(device)).item()
                )

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)

        joint_score, p_dir, v_dir, p_corr, v_corr, pred_up_pct, std_ratio = \
            compute_val_metrics(model, val_loader, device)

        elapsed = time.time() - start
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["joint_score"].append(joint_score)
        history["price_dir"].append(p_dir)
        history["price_corr"].append(p_corr)

        # ── Print once ────────────────────────────────────────────────────────
        print(f"{epoch:>6} | {train_loss:>8.5f} | {val_loss:>8.5f} | "
              f"{joint_score:>7.4f} | {p_dir:>6.3f} | {v_dir:>6.3f} | "
              f"{p_corr:>6.3f} | {v_corr:>6.3f} | {pred_up_pct:>4.0%} | "
              f"{std_ratio:>5.2f} | {elapsed:>4.1f}s")

        # ── Collapse detection with restart ───────────────────────────────────
        if epoch > WARMUP_EPOCHS:
            if pred_up_pct > 0.85 or pred_up_pct < 0.15 or std_ratio < 0.20:
                consecutive_collapse = getattr(training_state, 'collapse_ctr', 0) + 1
                training_state.collapse_ctr = consecutive_collapse
                if consecutive_collapse >= 5 and epoch < 30:
                    print(f"\n  ⚠ Collapse detected at epoch {epoch} "
                          f"(up%={pred_up_pct:.0%}, std_r={std_ratio:.2f}) — reloading init weights")
                    if os.path.exists(save_path):
                        model.load_state_dict(torch.load(save_path))
                    else:
                        model.apply(lambda m: m.reset_parameters()
                                    if hasattr(m, 'reset_parameters') else None)
                    for pg in optimizer.param_groups:
                        pg['lr'] = best["lr"] * 2.0
                    training_state.collapse_ctr = 0
            else:
                training_state.collapse_ctr = 0

        # ── Scheduler — once, after collapse check ────────────────────────────
        scheduler.step(joint_score)

        # ── Save on best joint score ───────────────────────────────────────────
        if joint_score > best_joint_score:
            best_joint_score = joint_score
            patience_ctr     = 0
            torch.save(model.state_dict(), save_path)
            print(f"         ✓ Saved  "
                  f"(p_dir={p_dir:.3f}, v_dir={v_dir:.3f}, "
                  f"p_corr={p_corr:.3f}, up%={pred_up_pct:.0%}, std_r={std_ratio:.2f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    trained_models[symbol] = {
        "best_joint_score": best_joint_score,
        "history":          history,
    }
    print(f"\n  Best joint score: {best_joint_score:.4f}")

print(f"\n── Training done for {len(trained_models)} symbols ──")
for s, r in trained_models.items():
    print(f"  {s}: best joint score = {r['best_joint_score']:.4f}")


════════════════════════════════════════════════════════════════════════
  Training — 2150
════════════════════════════════════════════════════════════════════════
  Best params: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.0017427913295284558, 'batch_size': 64, 'seq_len': 20, 'alpha': 6.0, 'beta': 1.5, 'gamma': 1.75, 'move_threshold': 0.4}

 Epoch |   TrLoss |   VaLoss |   Joint |   PDir |   VDir |     Pr |     Vr |   UP% |  StdR |     s
──────────────────────────────────────────────────────────────────────────────────────────
     1 |  2.72665 |  3.56064 |  0.3109 |  0.536 |  0.515 |  0.039 |  0.048 |  32% |  1.11 |  1.0s
         ✓ Saved  (p_dir=0.536, v_dir=0.515, p_corr=0.039, up%=32%, std_r=1.11)
     2 |  3.20823 |  4.65323 |  0.2193 |  0.537 |  0.519 |  0.021 |  0.033 |  57% |  1.54 |  1.0s
     3 |  3.07489 |  4.58220 |  0.2611 |  0.503 |  0.500 | -0.000 |  0.039 |  41% |  1.23 |  0.9s
     4 |  3.18411 |  4.63568 |  0.2652 |  0.510 |  0.500 |  0.005 |  0.040 

DIAGNOSTIC

In [8]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

def evaluate(actuals, preds, label):
    mae      = np.mean(np.abs(actuals - preds))
    rmse     = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc  = np.mean(np.sign(actuals) == np.sign(preds)) * 100

    corr       = np.corrcoef(actuals, preds)[0, 1]
    pred_std   = np.std(preds)
    actual_std = np.std(actuals)
    within_1pct = np.mean(np.abs(actuals - preds) < 1.0) * 100
    within_2pct = np.mean(np.abs(actuals - preds) < 2.0) * 100

    print(f"\n  [{label}]")
    print(f"  MAE             : {mae:.4f}%")
    print(f"  RMSE            : {rmse:.4f}%")
    print(f"  Pearson r       : {corr:.4f}")
    print(f"  Pred std        : {pred_std:.4f}  Actual std: {actual_std:.4f}")
    print(f"  Within ±1%      : {within_1pct:.1f}%")
    print(f"  Within ±2%      : {within_2pct:.1f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")

for symbol in symbols:
    if symbol not in prepared:
        print(f"\n  Skipping {symbol} — not in prepared dict.")
        continue
    if symbol not in trained_models:
        print(f"\n  Skipping {symbol} — no trained model found.")
        continue

    print(f"\n{'═'*55}")
    print(f"  Evaluating — {symbol}")
    print(f"{'═'*55}")

    best = best_params_all[symbol]
    data = prepared[symbol]
    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild test loader with best seq_len ─────────────────────────────────
    X_test_f, y_test_f, _ = create_sequences(          # ← unpack 3, discard w
        data["test_X_scaled"], data["test_y_scaled"], best["seq_len"], move_threshold
    )
    test_loader = DataLoader(
        TadawulDataset(X_test_f, y_test_f),            # no weights needed for eval
        batch_size=best["batch_size"],
        shuffle=False
    )

    # ── Load best model ───────────────────────────────────────────────────────
    checkpoint = torch.load(
        f"models/{symbol}/{symbol}_best_bilstm.pt",
        weights_only=True
    )

    inferred_hidden_size = checkpoint["lstm.weight_ih_l0"].shape[0] // 4
    inferred_num_layers  = sum(
        1 for k in checkpoint if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = inferred_hidden_size,
        num_layers  = inferred_num_layers,
        dropout     = best["dropout"]
    ).to(device)

    model.load_state_dict(checkpoint)
    model.eval()

    # ── Load target scaler ────────────────────────────────────────────────────
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        target_scaler = pickle.load(f)

    # ── Inference ─────────────────────────────────────────────────────────────
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:        # ← unpack 3
            preds   = model(X_batch.to(device)).cpu().numpy()
            actuals = y_batch.numpy()
            all_preds.append(preds)
            all_actuals.append(actuals)

    all_preds   = np.vstack(all_preds)
    all_actuals = np.vstack(all_actuals)

    # ── Inverse transform ─────────────────────────────────────────────────────
    all_preds   = target_scaler.inverse_transform(all_preds)
    all_actuals = target_scaler.inverse_transform(all_actuals)

    price_preds_pct    = all_preds[:, 0]
    price_actuals_pct  = all_actuals[:, 0]
    volume_preds_pct   = np.expm1(all_preds[:, 1])   * 100
    volume_actuals_pct = np.expm1(all_actuals[:, 1]) * 100

    # ── Metrics ───────────────────────────────────────────────────────────────
    evaluate(price_actuals_pct,  price_preds_pct,  "Price % Change")
    evaluate(volume_actuals_pct, volume_preds_pct, "Volume % Change")

    # ── Save predictions ──────────────────────────────────────────────────────
    results_df = pd.DataFrame({
        "actual_price_pct"    : price_actuals_pct,
        "predicted_price_pct" : price_preds_pct,
        "actual_volume_pct"   : volume_actuals_pct,
        "predicted_volume_pct": volume_preds_pct,
    })
    out_path = f"models/{symbol}/{symbol}_test_predictions.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\n  Predictions saved: {out_path}")

print(f"\n── Evaluation done for {len(symbols)} symbols ──")


EVALUATION ON TEST SET

═══════════════════════════════════════════════════════
  Evaluating — 2150
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 2.9778%
  RMSE            : 3.6426%
  Pearson r       : 0.0028
  Pred std        : 1.7677  Actual std: 1.7753
  Within ±1%      : 18.0%
  Within ±2%      : 37.4%
  Directional Acc : 51.61%

  [Volume % Change]
  MAE             : 61.5948%
  RMSE            : 104.6646%
  Pearson r       : 0.0776
  Pred std        : 4.4972  Actual std: 102.4346
  Within ±1%      : 1.2%
  Within ±2%      : 2.2%
  Directional Acc : 51.15%

  Predictions saved: models/2150/2150_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — 4260
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 2.3506%
  RMSE            : 2.8912%
  Pearson r       : 0.0063
  Pred std        : 1.2814  Actual std: 1.7025
  Within ±1%      : 24.6%
  Within ±2%

In [9]:
import pandas as pd
import numpy as np

for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0

    print(f"\n{'═'*55}")
    print(f"  {symbol} — Honest Model Baseline")
    print(f"{'═'*55}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size:")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 100.0]
    labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
    price_actual_abs = price_actual.abs()
    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean() * 100
            print(f"    {label:>8} moves : {acc:.2f}%  ({mask.sum()} samples)")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")


═══════════════════════════════════════════════════════
  2150 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : 0.0183   std: 1.7763
  Predicted mean : -2.6294   std: 1.7687

  Overall Dir Acc : 51.61%
  When actual UP  : 6.65%  (406 samples)
  When actual DOWN: 91.13%  (462 samples)

  % predicted UP  : 7.83%
  % actual UP     : 46.77%

  Dir Acc by Move Size:
       <0.5% moves : 56.22%  (233 samples)
      0.5-1% moves : 51.06%  (188 samples)
        1-2% moves : 51.21%  (248 samples)
        2-5% moves : 48.15%  (189 samples)
         >5% moves : 30.00%  (10 samples)

═══════════════════════════════════════════════════════
  4260 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0511   std: 1.7039
  Predicted mean : -2.0123   std: 1.2825

  Overall Dir Acc : 54.95%
  When actual UP  : 0.00%  (273 samples)
  When actual DOWN: 100.00%  (333 samples)

  % predicted UP  : 0.00%
  % ac

In [10]:
for symbol in symbols:
    df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    n = len(df)
    val_df  = df.iloc[int(n*0.70):int(n*0.85)]
    test_df = df.iloc[int(n*0.85):]
    
    print(f"\n{symbol}")
    print(f"  Val  UP%: {(val_df['price_pct_change'] > 0).mean():.2%}  ({val_df['datetime'].min().date()} → {val_df['datetime'].max().date()})")
    print(f"  Test UP%: {(test_df['price_pct_change'] > 0).mean():.2%}  ({test_df['datetime'].min().date()} → {test_df['datetime'].max().date()})")


2150
  Val  UP%: 46.33%  (2019-01-16 → 2022-09-13)
  Test UP%: 46.83%  (2022-09-14 → 2026-05-13)

4260
  Val  UP%: 47.12%  (2020-10-22 → 2023-08-06)
  Test UP%: 45.53%  (2023-08-07 → 2026-05-13)
